## Import ##

In [2]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [3]:
# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disable

## Functions ##

In [ ]:
# import cv2
# import os
# import numpy as np

# # Step 2: Function to Extract Frames from Video
# def extract_frames(video_path):
#     """Extract frames from a video at a specified frame rate."""
#     cap = cv2.VideoCapture(video_path)

#     # Check if video opened successfully
#     if not cap.isOpened():
#         print("Error: Could not open video.")
#         exit()

#     # List to store all frames
#     frames = []

#     # Read all frames
#     while True:
#         # Capture frame-by-frame
#         ret, frame = cap.read()
        
#         # If no frame is returned, break the loop
#         if not ret:
#             break

#         # Append the frame to the list
#         frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         frames.append(frame)

#     # Release the video capture object
#     cap.release()
#     return frames

In [4]:
# Step 3: Function to Extract Embeddings for a List of Frames
def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        for image in image_list:
            input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            features = model(input_tensor)
            embeddings.append(features.squeeze().cpu().numpy())
    return embeddings

## Process ##

In [6]:
# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
process_count = 0
for label_folder in os.listdir(video_folder):
    process_count += 1

    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in os.listdir(full_label_folder):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        image_list = []
        for image_file in os.listdir(full_sample_folder):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image_list.append(image)
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        video_labels.append(label)

with open('features_left_hand_frames_small.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

abc = 4

process_count:  1  , label:  664
process_count:  2  , label:  18
process_count:  3  , label:  22
process_count:  4  , label:  580
process_count:  5  , label:  489
process_count:  6  , label:  338
process_count:  7  , label:  444
process_count:  8  , label:  735
process_count:  9  , label:  610
process_count:  10  , label:  400
process_count:  11  , label:  263
process_count:  12  , label:  549
process_count:  13  , label:  61
process_count:  14  , label:  393
process_count:  15  , label:  483
process_count:  16  , label:  148
process_count:  17  , label:  165
process_count:  18  , label:  147
process_count:  19  , label:  537
process_count:  20  , label:  445
process_count:  21  , label:  443
process_count:  22  , label:  631
process_count:  23  , label:  169
process_count:  24  , label:  295
process_count:  25  , label:  86
process_count:  26  , label:  721
process_count:  27  , label:  378
process_count:  28  , label:  558
process_count:  29  , label:  83
process_count:  30  , label:

## Evaluation ##

In [6]:
with open('features_left_hand_frames_288_to_553.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [5]:
pickle_file11 = open('features_left_hand_frames_553_to_final.pickle', 'rb')
features11,labels11 = pickle.load(pickle_file11)
cc = 5

In [21]:
## Take average and 

pickle_file = open('/media/osero/SamsungSSD/pickles/features_left_hand_frames_288.pickle', 'rb')
features,labels = pickle.load(pickle_file)
average_features = []
frame_frequency = 5

for feature in features:
    sampled_features = feature[0::frame_frequency]
    average_features.append(np.mean(sampled_features, axis=0))
    ccc = 5

average_features = average_features[0:289]
labels = labels[0:289]
ccc = 5

KeyboardInterrupt: 

In [20]:
# Step 5: Train a Classifier on the Video Embeddings
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(average_features, labels, test_size=0.2)

# Initialize a k-NN classifier
knn = KNeighborsClassifier(n_neighbors = 1)

# Train the classifier
knn.fit(X_train, y_train)

# Predict on the test set and calculate accuracy
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

ValueError: Found input variables with inconsistent numbers of samples: [289, 287]

## Linear Classifier (MLP) ##

In [20]:
from torch.utils.data import DataLoader, Dataset

# Define a Dataset class for loading video features and labels
class VideoDataset(Dataset):
    def __init__(self, features, labels):
        self.labels = labels
        self.features = features
            
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx]).float(), torch.tensor(self.labels[idx]).long()


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
# import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define an MLP classifier
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLPClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [11]:
# Split features into training and testing sets
train_features, test_features, train_labels, test_labels  = train_test_split(average_features, labels, test_size=0.2)

# Create Dataset and DataLoader
train_dataset = VideoDataset(train_features, train_labels)
test_dataset = VideoDataset(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Define MLP classifier, loss, and optimizer, and move to GPU
input_dim = train_dataset[0][0].size(0)  # Get input dimension from a single feature
num_classes = len(set(labels))
mlp_classifier = MLPClassifier(input_dim=input_dim, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_classifier.parameters(), lr=0.001)
num_epochs = 10

In [12]:
# Training loop with batching
for epoch in range(num_epochs):
    mlp_classifier.train()
    running_loss = 0.0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        outputs = mlp_classifier(batch_features)
        
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")

/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasCreate(handle)`

In [5]:
# Training loop
for epoch in range(num_epochs):
    mlp_classifier.train()
    optimizer.zero_grad()
    outputs = mlp_classifier(train_features_tensor)
    loss = criterion(outputs, train_labels_tensor)
    loss.backward()
    optimizer.step()
    
print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# for idx, (features, labels) in enumerate(loop):
#     features = features.to(device)
#     labels = labels.to(device)

#     optimizer.zero_grad()
#     outputs = model(features)
#     loss = criterion(outputs, labels)

#     predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
#     correct = (predictions == labels).sum().item()
#     accuracy = correct / batch_size

#     loss.backward()
#     optimizer.step()

#     loop.set_description(f"Epoch [{epoch}/{num_epochs}]")
#     loop.set_postfix(loss=loss.item(), acc=accuracy)

/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_cl

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.

In [ ]:
# Evaluation
mlp_classifier.eval()
with torch.no_grad():
    test_outputs = mlp_classifier(test_features_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = accuracy_score(test_labels_tensor.cpu().numpy(), predicted.cpu().numpy())
    print(f"Test Accuracy: {accuracy:.4f}")

In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("PyTorch Version:", torch.__version__)

CUDA Available: True
CUDA Version: 11.7
PyTorch Version: 1.13.1


/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## https://github.com/purnasai/Dino_V2/blob/main/3.DinoV2_VS_ResnetClassification.ipynb ##

In [5]:
# Split features into training and testing sets
train_features, test_features, train_labels, test_labels  = train_test_split(average_features, labels, test_size=0.2)

# Create Dataset and DataLoader
train_dataset = VideoDataset(train_features, train_labels)
test_dataset = VideoDataset(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Define MLP classifier, loss, and optimizer, and move to GPU
input_dim = train_dataset[0][0].size(0)  # Get input dimension from a single feature
num_classes = len(set(labels))
mlp_classifier = MLPClassifier(input_dim=input_dim, num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
optimizer = optim.Adam(mlp_classifier.parameters(), lr=0.000001)
     

In [6]:
for epoch in range(6):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = mlp_classifier(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0

print('Finished Training')

/opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
